# 03 — Entertainment Content Strategy: Data Preparation

Sigue directo de [`01_eda.ipynb`](./01_eda.ipynb), donde quedaron anotadas las decisiones pendientes. Acá las ejecuto: corrijo el bug de datos (`duration` filtrado en `rating`), manejo los nulos como categoría `"Unknown"` en vez de eliminar filas, separo `duration` en `duration_min` (películas) y `n_seasons` (series), y construyo un campo de texto combinado (`content_soup`) para el recomendador content-based de `03_modeling.ipynb`.

El punto que más vueltas me dio fue el último: definir un target de "éxito" de forma honesta. El EDA ya había dejado claro que no hay métrica de audiencia en el dataset, así que en vez de inventar un proxy de éxito para todo el catálogo, acoté el problema predictivo a algo real y verificable — si una serie fue renovada para una segunda temporada o más. La renovación es una decisión de negocio real de Netflix, informada por datos de audiencia internos que nosotros no vemos, pero que sí queda reflejada en el catálogo.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

RAW_PATH = '../data/netflix_titles.csv'
TITLES_OUT = '../data/processed_titles.csv'
TV_SHOWS_OUT = '../data/processed_tv_shows.csv'

In [2]:
df = pd.read_csv(RAW_PATH)
print('Filas crudas:', df.shape)

Filas crudas: (8807, 12)


## 1. Corregir el bug `rating`/`duration`

Son las mismas 3 filas que aparecieron en el EDA: `rating` trae un valor de duración (`"74 min"`, etc.) y `duration` está nulo. El arreglo es directo — se mueve el valor a `duration` y `rating` queda como nulo genuino, que se imputa como `"Unknown"` en el paso siguiente junto con el resto de los nulos de `rating`.

In [3]:
bug_mask = df['rating'].astype(str).str.contains('min', na=False)
print(f'Filas afectadas por el bug: {bug_mask.sum()}')

df.loc[bug_mask, 'duration'] = df.loc[bug_mask, 'rating']
df.loc[bug_mask, 'rating'] = np.nan

assert df['duration'].isnull().sum() == 0, 'duration no debería tener nulos después del fix'
print('duration nulos tras el fix:', df['duration'].isnull().sum())
print('rating nulos tras el fix:', df['rating'].isnull().sum(), '(4 originales + 3 del bug)')

Filas afectadas por el bug: 3
duration nulos tras el fix: 0
rating nulos tras el fix: 7 (4 originales + 3 del bug)


## 2. Nulos → `"Unknown"`

Acá aplico la decisión que quedó tomada en el EDA: `director`, `cast`, `country` y `rating` se imputan como categoría explícita `"Unknown"`, no se eliminan filas.

In [4]:
for col in ['director', 'cast', 'country', 'rating']:
    df[col] = df[col].fillna('Unknown')

print(df[['director', 'cast', 'country', 'rating']].isnull().sum())

director    0
cast        0
country     0
rating      0
dtype: int64


Para `date_added` (10 nulos) no aplico el mismo criterio — no existe un "Unknown" razonable para una fecha. Se deja como fecha faltante (`NaT`) tras parsear; son pocas filas y no vale la pena forzar un valor artificial.

In [5]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
print('date_added nulos:', df['date_added'].isnull().sum())

date_added nulos: 10


## 3. Separar `duration` por tipo

`duration_min` solo tiene sentido para películas, `n_seasons` solo para series. Mezclarlas en una sola columna numérica sería comparar peras con manzanas — literalmente unidades distintas.

In [6]:
df['duration_min'] = np.where(
    df['type'] == 'Movie',
    pd.to_numeric(df['duration'].str.extract(r'(\d+)')[0], errors='coerce'),
    np.nan,
)
df['n_seasons'] = np.where(
    df['type'] == 'TV Show',
    pd.to_numeric(df['duration'].str.extract(r'(\d+)')[0], errors='coerce'),
    np.nan,
)

df[['type', 'duration', 'duration_min', 'n_seasons']].sample(6, random_state=42)

,type,duration,duration_min,n_seasons
4970,Movie,102 min,102.0,NaN
3362,Movie,63 min,63.0,NaN
5494,TV Show,3 Seasons,NaN,3.0
1688,TV Show,1 Season,NaN,1.0
1349,TV Show,1 Season,NaN,1.0
4862,TV Show,1 Season,NaN,1.0


## 4. `content_soup` — texto combinado para el recomendador

El recomendador content-based de `03_modeling.ipynb` compara títulos por similitud de texto, con TF-IDF más coseno. Para eso combino `description`, `listed_in` (géneros), `cast` y `director` en un solo campo de texto. Géneros y reparto se repiten a propósito — les da más peso implícito, para que dominen sobre palabras sueltas de la sinopsis en vez de perderse en el ruido.

In [7]:
def clean_list_field(text):
    """Junta un campo tipo 'A, B, C' en 'A B C' para que TF-IDF no trate
    'Julien Leclercq' como distinto de 'Leclercq Julien'."""
    if text == 'Unknown':
        return ''
    return text.replace(',', ' ').replace('  ', ' ')

df['content_soup'] = (
    df['description'].fillna('') + ' ' +
    (df['listed_in'] + ' ') * 2 +
    df['cast'].apply(clean_list_field) + ' ' +
    df['director'].apply(clean_list_field)
).str.lower()

df[['title', 'content_soup']].sample(2, random_state=42)

,title,content_soup
4970,"Game Over, Man!",three buddies with big dreams go from underach...
3362,Arsenio Hall: Smart & Classy,"in his first stand-up special, arsenio hall di..."


## 5. Target de "éxito": renovación de series

Acoto el subset a `type == 'TV Show'` y defino `renewed = 1` si la serie tiene 2 o más temporadas, `0` si tiene solo 1. Es una redefinición deliberada, y me parece más honesta que forzar un proxy de "éxito" sobre todo el catálogo. No se aplica a películas — no tienen una señal equivalente en este dataset, así que directamente quedan fuera de este target.

In [8]:
tv_shows = df[df['type'] == 'TV Show'].copy()
tv_shows['renewed'] = (tv_shows['n_seasons'] > 1).astype(int)

print('TV shows:', len(tv_shows))
print('Tasa de renovación:', tv_shows['renewed'].mean().round(4))
print(tv_shows['renewed'].value_counts())

TV shows: 2676
Tasa de renovación: 0.33
renewed
0    1793
1     883
Name: count, dtype: int64


## 6. Features para el modelo de renovación

La regla que me impuse acá fue estricta: solo features disponibles antes o al momento de estrenar la primera temporada. Nada que dependa de cuántas temporadas terminó teniendo la serie, porque eso sería filtrar la propia etiqueta disfrazada de feature. Uso `release_year`, `rating`, `country` (primer país listado), género principal (primer género listado) y la cantidad de géneros/países listados.

In [9]:
tv_shows['primary_country'] = tv_shows['country'].str.split(', ').str[0]
tv_shows['primary_genre'] = tv_shows['listed_in'].str.split(', ').str[0]
tv_shows['n_genres'] = tv_shows['listed_in'].str.split(', ').apply(len)
tv_shows['n_countries'] = tv_shows['country'].apply(lambda s: 0 if s == 'Unknown' else len(s.split(', ')))
tv_shows['has_director'] = (tv_shows['director'] != 'Unknown').astype(int)
tv_shows['description_len'] = tv_shows['description'].str.len()

feature_preview = ['title', 'release_year', 'rating', 'primary_country', 'primary_genre', 'n_genres', 'n_countries', 'has_director', 'description_len', 'renewed']
tv_shows[feature_preview].sample(5, random_state=42)

,title,release_year,rating,primary_country,primary_genre,n_genres,n_countries,has_director,description_len,renewed
2743,The Iliza Shlesinger Sketch Show,2020,TV-MA,United States,TV Comedies,1,1,0,135,0
7254,La Familia P. Luche,2012,TV-14,United States,International TV Shows,3,1,0,131,1
1306,Good Girls,2020,TV-MA,United States,Crime TV Shows,3,1,0,123,1
4640,Super Monsters Monster Party,2018,TV-Y,Unknown,Kids' TV,1,0,0,129,0
5537,The Miracle,2016,TV-14,South Korea,International TV Shows,3,1,0,149,0


Nota de leakage: descarto a propósito `n_seasons` y `duration` como features, porque son literalmente la variable con la que se construyó la etiqueta — usarlas sería hacer trampa sin darme cuenta. También descarto `date_added`: es la fecha de incorporación al catálogo, no de estreno, y no tiene ninguna relación causal con si la serie fue renovada o no.

## 7. Guardado de datasets procesados

Guardo dos salidas con propósitos distintos: `processed_titles.csv` es el catálogo completo (películas + series) con `content_soup`, pensado para el recomendador content-based; `processed_tv_shows.csv` es solo series, con `renewed` y las features de la sección 6, pensado para el modelo de clasificación.

Ninguna de las dos se versiona en git — se regeneran corriendo este notebook.

In [10]:
df.to_csv(TITLES_OUT, index=False)
tv_shows.to_csv(TV_SHOWS_OUT, index=False)

print('Guardado:', TITLES_OUT, df.shape)
print('Guardado:', TV_SHOWS_OUT, tv_shows.shape)

Guardado:

 ../data/processed_titles.csv (8807, 15)
Guardado: ../data/processed_tv_shows.csv (2676, 22)


## 8. Cómo queda el dato después de esta etapa

El bug quedó corregido: las 3 filas con duración filtrada en `rating` se arreglaron antes de cualquier análisis posterior. Los nulos de `director`, `cast`, `country` y `rating` se manejan como categoría explícita `"Unknown"`, así que no se perdió ninguna fila del catálogo en el camino. `duration` quedó separada en `duration_min` (películas) y `n_seasons` (series) — unidades distintas, columnas distintas — y `content_soup` combina sinopsis, géneros (con más peso) y reparto/director para el recomendador, sin necesitar ninguna etiqueta de éxito.

La parte que más me importa dejar clara: el target de "éxito" quedó redefinido de forma honesta como `renewed` (2+ temporadas), sobre el subset de 2.676 series, con una tasa de renovación del 33% — un desbalance moderado, no extremo. `n_seasons` y `duration` quedan excluidas de las features precisamente porque son la fuente literal de la etiqueta. Y las películas se quedan fuera de este modelo de "éxito" porque no hay ninguna señal equivalente a la renovación para ellas en este dataset; sí participan, en cambio, del recomendador.

Siguiente paso: `03_modeling.ipynb`, donde construyo el recomendador content-based sobre todo el catálogo y el modelo de clasificación para predecir renovación de series.